# 06 - Ejercicios Prácticos: Detección de Fraude en Pagos Digitales y Tarjetas con Isolation Forest

## Curso de Machine Learning: Modelos No Supervisados

En este cuaderno pondrás en práctica todo el flujo profesional de **Detección de Anomalías y Prevención de Fraude** utilizando **Isolation Forest** sobre un escenario bancario realista: la auditoría de **5,000 transacciones de comercio electrónico (*Card-Not-Present*)** donde se infiltran ataques de robo de cuenta (*Account Takeover*) y pruebas de tarjetas robadas (*Carding*).

> **Nota:** Encontrarás las soluciones completas y comentadas en `.soluciones/06_Ejercicios_Practicos_Solucion.ipynb`. Resuelve primero cada ejercicio por tu cuenta.

---

### Instrucciones Generales
1. Ejecuta la celda de generación del dataset transaccional de pagos digitales.
2. Escribe tu código en las celdas marcadas con `# TU CODIGO AQUI`.
3. Responde las preguntas de análisis financiero y forense en las celdas de Markdown.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    roc_auc_score, average_precision_score, 
    precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## Contexto del Dataset: Pasarela de Pagos E-Commerce (5,000 Transacciones, 8 Variables)

Cada fila representa una transacción con tarjeta en línea descrita por 8 variables de telemetría financiera:
- `monto_transaccion_usd`: Valor de la compra en dólares (USD).
- `ratio_monto_historico`: Razón entre el monto actual y el ticket promedio histórico del titular.
- `velocidad_tx_ultimas_2h`: Número de cargos realizados con la misma tarjeta en las últimas 2 horas.
- `distancia_ip_domicilio_km`: Distancia geográfica estimada entre la dirección IP del navegador y el domicilio del cliente (km).
- `cambio_dispositivo_horas`: Horas transcurridas desde que se registró el dispositivo actual (valores cercanos a 0 indican un dispositivo nuevo desconocido).
- `puntaje_riesgo_bin_pais`: Nivel de riesgo histórico del país emisor/receptor (0 a 100).
- `intentos_cvv_invalidos`: Número de intentos fallidos del código de seguridad CVV antes de aprobar.
- `hora_madrugada_score`: Índice de actividad en horario inusual de madrugada (0 a 10).

*Nota*: El objeto `y_fraude_auditoria` (`1` = Fraude confirmado, `0` = Legítima) se reserva exclusivamente para evaluar la calidad de nuestras detecciones al final.


In [ ]:
def cargar_dataset_pagos_ecommerce(seed=42):
    rng = np.random.default_rng(seed)
    n_legit = 4850
    # Transacciones legitimas (97%)
    legit = np.column_stack([
        rng.lognormal(4.0, 0.65, n_legit).clip(8, 750),     # monto_transaccion_usd
        rng.normal(1.05, 0.30, n_legit).clip(0.2, 2.5),     # ratio_monto_historico
        rng.poisson(1.1, n_legit).clip(1, 4),               # velocidad_tx_ultimas_2h
        rng.exponential(15.0, n_legit).clip(0.5, 120.0),    # distancia_ip_domicilio_km
        rng.normal(720.0, 250.0, n_legit).clip(48.0, 2000), # cambio_dispositivo_horas
        rng.normal(18.0, 7.0, n_legit).clip(2.0, 45.0),     # puntaje_riesgo_bin_pais
        rng.choice([0, 1], size=n_legit, p=[0.95, 0.05]),   # intentos_cvv_invalidos
        rng.normal(2.0, 1.2, n_legit).clip(0.0, 6.0)        # hora_madrugada_score
    ])
    
    # Fraudes (3% = 150 casos: mezcla de Account Takeover de alto monto y Carding rapido)
    n_fraude = 150
    fraude = np.column_stack([
        rng.uniform(1200.0, 5400.0, n_fraude),              # monto_transaccion_usd
        rng.uniform(5.5, 22.0, n_fraude),                   # ratio_monto_historico
        rng.integers(5, 16, n_fraude),                      # velocidad_tx_ultimas_2h
        rng.uniform(600.0, 8500.0, n_fraude),               # distancia_ip_domicilio_km
        rng.uniform(0.05, 4.5, n_fraude),                   # cambio_dispositivo_horas (recien cambiado)
        rng.uniform(55.0, 98.0, n_fraude),                  # puntaje_riesgo_bin_pais
        rng.choice([1, 2, 3, 4], size=n_fraude, p=[0.1, 0.4, 0.3, 0.2]), # intentos_cvv
        rng.uniform(6.5, 10.0, n_fraude)                    # hora_madrugada_score
    ])
    
    X = np.vstack([legit, fraude])
    y = np.array([0]*n_legit + [1]*n_fraude)
    perm = rng.permutation(len(X))
    cols = [
        'monto_transaccion_usd', 'ratio_monto_historico', 'velocidad_tx_ultimas_2h',
        'distancia_ip_domicilio_km', 'cambio_dispositivo_horas', 'puntaje_riesgo_bin_pais',
        'intentos_cvv_invalidos', 'hora_madrugada_score'
    ]
    ids = [f'PAY_{i:04d}' for i in range(1, len(X) + 1)]
    return pd.DataFrame(X[perm], index=ids, columns=cols), pd.Series(y[perm], index=ids, name='es_fraude')

df_pagos, y_fraude_auditoria = cargar_dataset_pagos_ecommerce()
print(f"Dimensiones de df_pagos: {df_pagos.shape} | Tasa real de fraude oculta: {y_fraude_auditoria.mean()*100:.2f}%")
df_pagos.head()


---
## Ejercicio 1: Entrenamiento de Isolation Forest y Análisis de Puntajes de Camino

1. Instancia y entrena un modelo `IsolationForest(n_estimators=200, max_samples=256, contamination=0.03, random_state=42)` sobre `df_pagos` (sin pasar `y_fraude_auditoria`).
2. Calcula las predicciones binarias (convirtiendo `-1` a `1` [Alerta] y `+1` a `0` [Normal]) y el puntaje continuo de riesgo `-modelo.score_samples(df_pagos)`.
3. Grafica el histograma de los puntajes de riesgo separando por el color real (`y_fraude_auditoria`) y evalúa el **ROC-AUC** y el **PR-AUC (`average_precision_score`)**.


In [ ]:
# TU CODIGO AQUI: Entrenamiento de IsolationForest, histograma de scores y métricas AUC



---
## Ejercicio 2: Sensibilidad al Hiperparámetro `max_samples` (Efecto Masking)

1. Evalúa cómo cambia el **PR-AUC (`average_precision_score`)** cuando entrenas `IsolationForest` variando `max_samples` en la lista `[32, 64, 128, 256, 512, 1024, 2048, 5000]`.
2. Grafica la curva de `max_samples` vs. `PR-AUC`.
3. **Pregunta:** ¿Qué ocurre con la capacidad de detectar a los defraudadores cuando `max_samples=5000` (todo el dataset) en comparación con `max_samples=256`?


In [ ]:
# TU CODIGO AQUI: Experimento variando max_samples y gráfica de PR-AUC



---
## Ejercicio 3: Explicabilidad Forense Local de la Transacción Más Riesgosa

1. Identifica el ID de la transacción (`PAY_XXXX`) que obtuvo el **mayor puntaje de anomalía** en todo el dataset.
2. Aplica el método de atribución contrafactual: reemplaza una por una cada variable de esa transacción por la mediana del grupo normal y mide cuánto cae el puntaje de anomalía.
3. Grafica un diagrama de barras horizontales con el porcentaje de culpa de cada variable para esa transacción.


In [ ]:
# TU CODIGO AQUI: Identificación del caso top 1 y desglose de atribución por variable



---
## Ejercicio 4: Optimización del Umbral por Presupuesto Financiero

Supón que cada fraude no detectado (**FN**) le cuesta a la pasarela de pagos **$1,200 USD** en contracargos (*chargebacks*), y cada alerta enviada a bloqueo/verificación manual (**TP + FP**) tiene un costo administrativo y de fricción de **$25 USD**.
1. Explora 100 umbrales posibles sobre el puntaje de riesgo y calcula el costo financiero total en cada umbral.
2. Encuentra el umbral óptimo que minimiza la pérdida en dólares y calcula la **Precisión**, el **Recall** y el **Ahorro Neto en USD** frente a no tener modelo.


In [ ]:
# TU CODIGO AQUI: Curva de costo en USD y selección del umbral financiero óptimo



---
## Ejercicio 5: Extracción de Reglas Antifraude mediante Árbol Sustituto

1. Entrena un `DecisionTreeClassifier(max_depth=3, random_state=42)` usando `df_pagos` como entrada y las alertas de tu `IsolationForest` como variable objetivo.
2. Imprime las reglas en texto (`export_text`) y visualiza el árbol con `plot_tree`.
3. Redacta las 2 reglas principales que un ingeniero de software podría programar inmediatamente en el motor de pagos en vivo.


In [ ]:
# TU CODIGO AQUI: Árbol de decisión sustituto y visualización



**Tus Conclusiones y Reglas de Motor Antifraude (Ejercicio 5):**
- **Regla 1:** *...*
- **Regla 2:** *...*
